# Train Your First Neural Network
**AI Learning Series, Day 11 (Part 3 of the Deep Learning block)**: Tech Boy Deepak

In Video A we **drew** a small neural network: 2 inputs (study hours, sleep hours) → 2 hidden neurons (ReLU) → 1 output (sigmoid).
That network had **9 knobs** (weights and biases), and we *guessed* them ourselves.
In Video B we saw **how a network learns**: forward pass → loss → backpropagation → gradient-descent update → repeat.

Today the computer does all of that for us. We build the same network in **Keras** and let it learn its 9 knobs from data.

**How to use this notebook**
- Click **Runtime → Run all**, or run the cells one by one with **Shift + Enter**.
- No files to upload or download. Everything is created inside the notebook.
- The whole notebook takes about 2–3 minutes on the free Colab CPU. No GPU is needed.
- Your numbers may come out a little different from the video. That's normal (see the note at the end).

## Block 1: Get our tools ready
- **numpy** is a calculator for lots of numbers at once.
- **matplotlib** draws graphs.
- **keras** is the neural-network library. It comes pre-installed with TensorFlow on Colab.

In [ ]:
import numpy as np                 # numbers and tables of numbers
import matplotlib.pyplot as plt    # graphs
import keras                       # the neural-network library (Keras 3, runs on TensorFlow)

print("Keras version:", keras.__version__)

## Block 2: Make our class data (features + label)
We create **500 pretend students**. For each student we have:
- **Features** (the inputs): study hours per day (0–10) and sleep hours per night (3–10)
- **Label** (the answer key): Pass = 1, Fail = 0

To create the data we use a hidden rule: **marks = 7 × study + 3 × sleep − 5 + luck**, and **Pass if marks ≥ 35**.
"Luck" means good days and bad days, so even a perfect model can't be right 100% of the time.

**Important:** the network never sees this rule. It only sees the 500 examples and must learn the pattern from them, just like a real dataset.

In [ ]:
rng = np.random.default_rng(42)       # fixed seed, so everyone gets the same students
n_students = 500

study = rng.uniform(0, 10, n_students).round(1)   # hours studied per day
sleep = rng.uniform(3, 10, n_students).round(1)   # hours slept per night
luck  = rng.normal(0, 8, n_students)              # a few marks up or down

marks  = 7 * study + 3 * sleep - 5 + luck
passed = (marks >= 35).astype(int)                # label: 1 = Pass, 0 = Fail

X = np.column_stack([study, sleep])   # features: 2 columns
y = passed                            # label: 1 column

print("First 5 students:")
print(" study  sleep  ->  label")
for i in range(5):
    print(f"{X[i, 0]:6.1f} {X[i, 1]:6.1f}  ->  {'Pass' if y[i] == 1 else 'Fail'}")
print(f"\nTotal students: {len(y)}, passed: {y.sum()} ({y.mean():.0%})")

## Block 3: Training data vs testing data
It's just like exams: we **teach** the model with 400 students (**training data**) and keep 100 students hidden for the **final exam** (**testing data**).
The model never sees the test students while it learns.

We also make the numbers smaller: **hours ÷ 10** (6 hours → 0.6). In the last video, attendance 60% became 0.6 for the same reason. Small inputs make learning smoother.

In [ ]:
X_scaled = X / 10                              # 6 hours -> 0.6

X_train, y_train = X_scaled[:400], y[:400]     # 400 students to learn from
X_test,  y_test  = X_scaled[400:], y[400:]     # 100 students kept hidden for the final exam

print("Training students:", len(X_train))
print("Testing students :", len(X_test))

## Block 4: Build the network we drew
- `keras.Input(shape=(2,))`: **2 inputs** (study, sleep)
- `Dense(2, activation="relu")`: the **hidden layer**, 2 circles using **ReLU**. "Dense" means every input is connected to every neuron (fully connected).
- `Dense(1, activation="sigmoid")`: the **output layer**, 1 circle using **sigmoid**, which gives the pass probability.

Look at the **Param #** column: **6 + 3 = 9**. That's 4 weights + 2 biases in the hidden layer and 2 weights + 1 bias in the output. These are **the same 9 knobs from Video A**!

In [ ]:
keras.utils.set_random_seed(42)   # fixes the random starting knobs, so results repeat

model = keras.Sequential([
    keras.Input(shape=(2,)),                                          # 2 inputs: study, sleep
    keras.layers.Dense(2, activation="relu", name="hidden_layer"),    # 2 hidden neurons (ReLU)
    keras.layers.Dense(1, activation="sigmoid", name="output_layer"), # 1 output neuron (sigmoid)
])

model.summary()   # look for "Trainable params: 9"

## Block 5: Compile (choose the loss, the gradient-descent version and the learning rate)
- `loss="binary_crossentropy"`: the **log loss** from Video B, for pass/fail answers.
- `optimizer=SGD(learning_rate=0.1)`: **gradient descent**, stepping opposite to the slope. The **learning rate = step size = 0.1**, the same 0.1 we used for Priya.
- `metrics=["accuracy"]`: also show the % of correct answers. Remember: we *train* on loss and *report* accuracy.

In [ ]:
model.compile(
    loss="binary_crossentropy",                          # log loss: how wrong? (Video B)
    optimizer=keras.optimizers.SGD(learning_rate=0.1),   # gradient descent, step size 0.1
    metrics=["accuracy"],                                # also report % correct
)

## Block 6: Fit (the training loop: epochs and batches)
`model.fit()` is the loop from Video B: **forward → loss → backprop → update → repeat**.
- `validation_split=0.2`: the last 20% of our 400 training students (80 students) are kept aside as a **mock test**, checked after every epoch. So **320 students** actually train.
- `batch_size=32`: look at 32 students, average their blame, then update the 9 knobs **once**. 320 ÷ 32 = **10 batches per epoch**. You'll see `10/10` on every line.
- `epochs=100`: go through all 320 students **100 times**. Total updates = 100 × 10 = **1,000**.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=100,             # 100 full rounds through the training students
    batch_size=32,          # 32 students per knob update
    validation_split=0.2,   # 80 of the 400 kept aside as a mock test
    verbose=2,              # print one line per epoch
)

## Block 7: Look at the loss curves
- **Training loss**: how wrong the model is on the students it learns from.
- **Validation loss**: how wrong it is on the 80 mock-test students it does **not** learn from.

Both going down together means the model is genuinely learning, not just memorising.

In [ ]:
epochs_axis = range(1, len(history.history["loss"]) + 1)   # epoch 1, 2, 3, ...
plt.plot(epochs_axis, history.history["loss"], label="training loss")
plt.plot(epochs_axis, history.history["val_loss"], label="validation loss (mock test)")
plt.xlabel("epoch")
plt.ylabel("loss (lower = better)")
plt.title("Loss goes down as the network learns")
plt.legend()
plt.show()

for ep in [1, 10, 50, 100]:
    print(f"After epoch {ep:3d}: training loss = {history.history['loss'][ep - 1]:.3f}, "
          f"training accuracy = {history.history['accuracy'][ep - 1]:.0%}")

## Block 8: Final exam (evaluate on the test data)
Now we check the 100 students the model has **never seen**. This is the honest score.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss     : {test_loss:.3f}")
print(f"Test accuracy : {test_accuracy:.0%}  (on 100 students the model never saw)")

## Block 9: Predict for Sita, Ravi and Kiran
Remember Video A? With our **guessed** knobs, the network said Sita (6 h study, 3 h sleep) had only **38%** chance to pass. But she **actually passed**.
Now the knobs are **learned from data**. Let's ask again. (We divide by 10, exactly like the training data.)

In [ ]:
students = {"Sita": (6, 3), "Ravi": (4, 7), "Kiran": (1, 8)}

for name, (s, l) in students.items():
    prob = model.predict(np.array([[s, l]]) / 10, verbose=0)[0][0]   # same ÷10 as training
    verdict = "PASS" if prob >= 0.5 else "FAIL"
    print(f"{name:5s}: studied {s} h, slept {l} h  ->  {prob:.0%} pass  ->  {verdict}")

**Try it yourself:** change the two numbers below and run the cell again.

In [ ]:
my_study, my_sleep = 5, 7      # <- change these
prob = model.predict(np.array([[my_study, my_sleep]]) / 10, verbose=0)[0][0]
print(f"Studied {my_study} h, slept {my_sleep} h  ->  {prob:.0%} pass")

**The 9 knobs the machine found by itself.** In Video A we typed in 1, 0.5, −6 and so on. Here are the learned values.
(They're for the ÷10 inputs, so don't compare them number-for-number with Video A.)

In [ ]:
w_hidden, b_hidden = model.get_layer("hidden_layer").get_weights()
w_out, b_out = model.get_layer("output_layer").get_weights()

print("Hidden layer (4 weights + 2 biases):")
print(f"  study -> h1: {w_hidden[0, 0]:5.2f}   sleep -> h1: {w_hidden[1, 0]:5.2f}   h1 bias: {b_hidden[0]:5.2f}")
print(f"  study -> h2: {w_hidden[0, 1]:5.2f}   sleep -> h2: {w_hidden[1, 1]:5.2f}   h2 bias: {b_hidden[1]:5.2f}")
print("Output layer (2 weights + 1 bias):")
print(f"  h1 -> output: {w_out[0, 0]:5.2f}   h2 -> output: {w_out[1, 0]:5.2f}   output bias: {b_out[0]:5.2f}")

## Experiment 1: Learning rate too small, about right, too big
Same network, same data, same starting knobs. We only change the **step size**.
- **0.001**: tiny steps, so learning is very slow
- **0.1**: about right
- **20.0**: giant steps, so it jumps over the valley and the loss zigzags

(Tip: always write the learning rate with a decimal point, e.g. `20.0` and not `20`. Keras gives an error for a whole number.)

In [ ]:
def build_small_model():
    return keras.Sequential([
        keras.Input(shape=(2,)),
        keras.layers.Dense(2, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])

lr_results = {}
for lr in [0.001, 0.1, 20.0]:            # too small, about right, too big
    keras.utils.set_random_seed(42)      # same starting knobs: a fair race
    m = build_small_model()
    m.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.SGD(learning_rate=lr),
              metrics=["accuracy"])
    h = m.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=0)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    lr_results[lr] = h.history["loss"]
    print(f"learning rate {lr:>6}: loss after epoch 1 = {h.history['loss'][0]:.3f}, "
          f"after epoch 100 = {h.history['loss'][-1]:.3f}, test accuracy = {acc:.0%}")

for lr, losses in lr_results.items():
    plt.plot(range(1, len(losses) + 1), losses, label=f"learning rate = {lr}")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Step size matters")
plt.legend()
plt.show()

## Experiment 2: Too many epochs, and memorising instead of learning (overfitting)
Now we make things easy to memorise: a **big network** (thousands of knobs), only **30 students** to learn from, and **500 epochs**.
Watch the two curves:
- **Training loss** keeps going down, almost to 0. The model has memorised those 30 students (బట్టీ).
- **Validation loss** goes down at first, then **goes back up**. The model is getting *worse* on new students.

This model uses **Adam**, the "smarter gradient descent" we name-dropped in Video B.

In [ ]:
keras.utils.set_random_seed(42)
big_model = keras.Sequential([
    keras.Input(shape=(2,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
big_model.compile(loss="binary_crossentropy",
                  optimizer=keras.optimizers.Adam(learning_rate=0.01),
                  metrics=["accuracy"])
print("Knobs in the big model:", big_model.count_params())

few_X, few_y     = X_train[:30], y_train[:30]       # only 30 students to learn from
check_X, check_y = X_train[320:], y_train[320:]     # 80 other students as the mock test

h_big = big_model.fit(few_X, few_y, epochs=500, batch_size=32,
                      validation_data=(check_X, check_y), verbose=0)

train_loss, val_loss = h_big.history["loss"], h_big.history["val_loss"]
best = int(np.argmin(val_loss))
ep_axis = range(1, len(train_loss) + 1)
plt.plot(ep_axis, train_loss, label="training loss (30 students)")
plt.plot(ep_axis, val_loss, label="validation loss (80 new students)")
plt.axvline(best + 1, linestyle="--", color="gray", label=f"best epoch = {best + 1}")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Too many epochs: memorising, not learning")
plt.legend()
plt.show()

print(f"Best validation loss {val_loss[best]:.3f} at epoch {best + 1}")
print(f"After 500 epochs: training loss {train_loss[-1]:.4f} "
      f"(training accuracy {h_big.history['accuracy'][-1]:.0%}), "
      f"validation loss {val_loss[-1]:.3f}")

## Optional: a slightly bigger network
Two hidden layers, 16 and 8 neurons, everything else the same as our main model.
Does it beat our 9-knob network on this simple data? Run it and see.

In [ ]:
keras.utils.set_random_seed(42)
bigger = keras.Sequential([
    keras.Input(shape=(2,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
bigger.compile(loss="binary_crossentropy",
               optimizer=keras.optimizers.SGD(learning_rate=0.1),
               metrics=["accuracy"])
bigger.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=0)

loss_b, acc_b = bigger.evaluate(X_test, y_test, verbose=0)
print(f"Bigger network: {bigger.count_params()} knobs, test accuracy {acc_b:.0%}, test loss {loss_b:.3f}")
print(f"Our 9-knob one: 9 knobs, test accuracy {test_accuracy:.0%}, test loss {test_loss:.3f}")

## What we did (block → what we learned earlier)
| Block | Code | Means |
|---|---|---|
| 2 | `X`, `y` | features + label (the answer key) |
| 3 | `X_train`, `X_test` | training vs testing |
| 4 | `Dense(2, relu)`, `Dense(1, sigmoid)` | the circles we drew; ReLU and sigmoid; 9 knobs |
| 5 | `compile(loss, optimizer, learning_rate)` | log loss + gradient descent + step size |
| 6 | `fit(epochs, batch_size)` | forward → loss → backprop → update, repeated |
| 8 | `evaluate` | the final exam on unseen students |
| 9 | `predict` | asking the trained model about Sita |

**Homework (comment your results!):**
1. In Block 6, change `epochs=100` to `epochs=5`. What test accuracy do you get?
2. Change `batch_size=32` to `batch_size=320`. How many batches per epoch does the output show now?
3. In Experiment 1, try `learning_rate=1.0`. Too big, or still OK?

**Why are my numbers slightly different from the video?** Training starts from random knobs and shuffles the students every epoch. We fixed the seed, but different computers (CPU vs GPU) and library versions can still give slightly different decimals. If accuracy gets stuck around 68–70% and every student gets the same %, the tiny network got unlucky starting knobs (its ReLU neurons switched off). Change the seed in Block 4 (e.g. 42 → 7) and run again.